In [0]:
%run ./P5_00_Config

In [0]:
import pandas as pd
import matplotlib.pyplot as plt

In [0]:
df_gold = spark.table(GOLD_TABLE).toPandas()
df_gold["ds"] = pd.to_datetime(df_gold["ds"])

In [0]:
import mlflow

client = mlflow.tracking.MlflowClient()

# Search runs across all relevant experiments
experiment_ids = ["2440334981483612", "2495525534209105"]

runs = client.search_runs(
    experiment_ids=experiment_ids,
    order_by=["metrics.mae ASC"]
)

for run in runs:
    mae = run.data.metrics.get("mae", None)
    if mae is not None:
        print(f"Run: {run.info.run_name} | Family: {run.data.params.get('family', 'N/A')} | MAE: {mae:.2f}")

In [0]:
import mlflow

client = mlflow.tracking.MlflowClient()

# List all experiments
experiments = client.search_experiments()
for exp in experiments:
    print(f"ID: {exp.experiment_id} | Name: {exp.name}")

In [0]:
#Split by model
df_prophet = df_gold[df_gold["model"] == "prophet"].sort_values("ds")
df_xgboost = df_gold[df_gold["model"] == "xgboost"].sort_values("ds")

#Plot
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(df_prophet["ds"], df_prophet["yhat"], label="Prophet", color="steelblue", linewidth=1.5)
ax.plot(df_xgboost["ds"], df_xgboost["yhat"], label="XGBoost", color="darkorange", linewidth=1.5)

ax.set_title("Demand Forecast Comparison - AUTOMOTIVE")
ax.set_xlabel("Date")
ax.set_ylabel("Sales")
ax.legend()
plt.tight_layout()
plt.show()